#  Proyecto Final Semillero De Inteligencia Artificial: Agentes especializados para el Área de Recursos Humanos— Agentes con Lanching

Departamento de Recursos Humanos — Patito S.A. el dpt. de rrhh recibe siempre las mismas preguntas 

> *"¿Qué cubre el seguro médico corporativo y cómo agrego a un familiar como dependiente?*  
> *"¿Cuántos días de vacaciones me corresponden al año y cómo solicito un permiso no remunerado?"*  
> *"¿Cómo funciona el programa de referidos y qué pasos incluye el onboarding de un nuevo ingreso?"*

Nuestra empresa **Telematicos S.A** conformado por **Diaz Ariel, Marin Geoshimar, Vallejo Lady** , fue contratado para optimizar el dpt. de RRHH hay que construir **agentes con lanching y como orquestador usar google gemini**.
### Lo que vas a construir

| agente  | que hace  | tecnologia |
|---|---|---|
| **conocimiento (rag)** | responsable de las politicas de seguro medico , proceso de seleccion , vaciones etc. |embeddings gemini + chroma
| **multimodal de imagen**  | lee factura y datos | gemini con vision |
| **Accion (regla)**  | registra todo en un txt. | #tools  |
| **Orquestador**  |decide que agente usa en cada consulta  |create_agente|

# 1 INSTALAMOS LAS LIBRERIAS NECESARIAS 
En este caso instalamos **LangChain** como framework para conectar la aplicación con **Google Gemini** (LLM y embeddings), **langchain-community** para acceder a integraciones adicionales, **langchain-chroma** y **ChromaDB** para almacenar y consultar embeddings en una base de datos vectorial, **Pillow** para el procesamiento básico de imágenes y **Pandas** para la manipulación y análisis de datos.

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-chroma chromadb pillow pandas
print("Instalacion completa.")


# 2  ☁️Ejecución en la Nube con Google Gemini
Google Gemini es el modelo de ia de google que utilizamos en este proyecto ya que es un modelo que utiliza recursos en la nube para ejecutarse sin usar recursos propios del ordenador, y se accede a ella usando un api key o una llave de acceso unica que hace el puente de nuestro proyecto con lo recursos de google.
### 🔑 ¿Cómo obtener tu API Key?

1. Ve a [Google AI Studio](https://aistudio.google.com/apikey)
2. Inicia sesión con tu cuenta de Google
3. Haz clic en **"Create API Key"**
4. Copia la clave generada y pégala en la celda de abajo
 # uso de librerias de google 
 **import os:** Importa la librería de Python que permite trabajar con el sistema operativo, por ejemplo, leer variables de entorno (como una API Key) o manejar archivos y carpetas.
**from langchain_google_genai import ChatGoogleGenerativeAI:** Importa la clase que permite usar el modelo de lenguaje de Google Gemini para generar respuestas en un chat.
**from langchain_google_genai import GoogleGenerativeAIEmbeddings:** Importa la clase que permite convertir texto en embeddings,

In [1]:
# MODO NUBE (POR DEFECTO) — Embeddings y LLM con Google Gemini - esto es la parte del lanching
import os
from dotenv import load_dotenv
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

load_dotenv()

MODELO_LLM = "gemini-3.1-flash-lite"
MODELO_EMBEDDING = "models/gemini-embedding-001"

llm = ChatGoogleGenerativeAI(
    model=MODELO_LLM,
    temperature=0
)

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING
)

# Print: si esta imprime este mensaje, estás conectado.
print(
    llm.invoke(
        "Responde únicamente: 'Gemini conectado' y nada más."
    ).content
)


[{'type': 'text', 'text': 'Gemini conectado', 'extras': {'signature': 'EjQKMgERTTIPUcuweiSVypIreQBXVB0641JsenHB4vYWgP5jhOosCRduYKhtcTtGT5RjMtPT'}}]


## 3. VAMOS A CREAR EL PRIMER AGENTE: AGENTE DE BENEFICIOS Y COMPENSACIÓN
el primer paso es el agente va a leer la base de conocimeintos el agente debe de responder en base al documento nada mas 
librerias : **from pathlib import Path** importa la clase **Path** de la librería pathlib, que sirve para trabajar con archivos y carpetas de una forma más sencilla.

In [2]:
# 1.el primer paso es importar librerias 
from pathlib import Path 

# 2. creamos las variables de documentos y politicas y ponemos su ruta ademas si no esta el doc cargado , lo escribimos 
DOC_PATH = "01_Beneficios_Compensaciones.txt"
POLITICA = """PATITO S.A.
MANUAL DE BENEFICIOS Y COMPENSACIONES
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Beneficios y Compensaciones

1. SEGURO MÉDICO CORPORATIVO
1.1 Cobertura: consultas médicas, hospitalización, emergencias, exámenes de laboratorio y
    medicamentos según el plan. Incluye atención ambulatoria y cobertura dental básica.
1.2 Dependientes: el colaborador puede inscribir a cónyuge o pareja e hijos.
1.3 Cómo agregar un dependiente:
    - Completar el formulario de inscripción de dependientes en el portal de RR. HH.
    - Adjuntar el documento que acredite el vínculo (acta de matrimonio/unión o partida de
      nacimiento) y copia del documento de identidad del dependiente.
    - Enviar la solicitud dentro de los primeros 30 días desde el ingreso o desde el evento
      (matrimonio, nacimiento). Fuera de ese plazo, se espera al periodo de inscripción anual.

2. BONOS
- Bono por desempeño anual según evaluación.
- Bono por cumplimiento de metas del área (cuando aplique).

3. OTROS BENEFICIOS
- Día libre de cumpleaños.
- Capacitación y apoyo educativo.
- Modalidad híbrida según el puesto.

4. COMPENSACIÓN
La estructura salarial considera el rol, la banda salarial y el mercado. Las revisiones
salariales se realizan una vez al año."""

# 3.Guardar la instancia path en la variable doc_path_obj
doc_path_obj = Path(DOC_PATH)

# 4. Verificar si el documento existe o crearlo si no está presente
if not doc_path_obj.exists():
    doc_path_obj.write_text(POLITICA, encoding="utf-8")
    print("docpack creado")
else:
    print("El archivo ya existe. Procediendo a la lectura...")

# 5.Utilizar el comando .read_text() para leer el doc y guardarlo en la variable politica
politica = doc_path_obj.read_text(encoding="utf-8")

# 6. Impresión de métricas requeridas de los primeros 400 caracteres
print(f"Caracteres totales: {len(politica)}")
print(" " * 60)
print(politica[:400])

El archivo ya existe. Procediendo a la lectura...
Caracteres totales: 1302
                                                            
PATITO S.A.
MANUAL DE BENEFICIOS Y COMPENSACIONES
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Beneficios y Compensaciones

1. SEGURO MÉDICO CORPORATIVO
1.1 Cobertura: consultas médicas, hospitalización, emergencias, exámenes de laboratorio y
    medicamentos según el plan. Incluye atención ambulatoria y cobertura dental básica.
1.2 Dependientes: e


## 4. chunkings +  Embeddings + ChromaDB 
**chunking:** El bloque de código que utiliza se utiliza para segmentar la informacion y se lo divide por temas 
**embeddings:**  Esta sección convierte los fragmentos de texto plano en vectores matemáticos 
**chromadb:** Esta seccion permite guardar los vectores matematicos en una base de datos 


In [3]:
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    
    # se crea la variable cabecera donde se guardara la lista de los temas buscados por segmentos 
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))

    chunks = []

    for i, m in enumerate(cabeceras):
        ini = m.start()
        # Si no es el último tema, el final es donde empieza el siguiente. Si es el último, va hasta el fin del texto.
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)

        # Extraemos la sección correspondiente y limpiamos espacios en blanco innecesarios
        parte = texto[ini:fin].strip()
        chunks.append(parte)

    return chunks

# 1. Obtenemos los chunks del manual de beneficios y compensaciones y la guardamos en la variable chunk_beneficios 

chunks_beneficios = chunkear_por_tema(politica)

# 2. en la varible chunks adicionales se guardan documentos auxiliares, si llegaran a existir 
chunks_adicionales = [] 

# Unimos los documentos procesados
chunks = chunks_beneficios + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks)}")
print("=" * 60)

for i, c in enumerate(chunks):
    # Reemplazamos los saltos de línea internos por espacios solo para la previsualización del print
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")
    
# Cada chunk se embebe con Gemini y se guarda en Chroma
# 1. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_chunks = [{"seccion": i + 1, "fuente": DOC_PATH} for i in range(len(chunks))]

# 2. Inicializamos el vectorstore en Chroma con la configuración e inserción de datos corregida
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    metadatas=metadatos_chunks,
    collection_name="beneficios_y_compensaciones"
)

# 3. Configuramos retriever para extraer los 2 mejores resultados (k=2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Base de conocimiento embebida en Chroma con embeddings de Gemini.")


C:\Users\GEO\AppData\Local\Temp\ipykernel_4864\2591253291.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Total de chunks creados: 4
Chunk 1: 1. SEGURO MÉDICO CORPORATIVO 1.1 Cobertura: consultas médicas, hospita...
Chunk 2: 2. BONOS - Bono por desempeño anual según evaluación. - Bono por cumpl...
Chunk 3: 3. OTROS BENEFICIOS - Día libre de cumpleaños. - Capacitación y apoyo ...
Chunk 4: 4. COMPENSACIÓN La estructura salarial considera el rol, la banda sala...
Base de conocimiento embebida en Chroma con embeddings de Gemini.


## 5. Agente de conocimiento uno (RACK)  Beneficios y compensaciones


In [4]:
PROMPT_CONOCIMIENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre el manual de beneficios y compensaciones.
Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""
def responder_politica(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento y genera la respuesta."""
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])
    
    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    contenido = msg.content
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        return "".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict) and bloque.get("type") == "text"
        )
    return str(contenido)
# Prueba
print(responder_politica("¿dime sobre los dependientes ?"))

Sobre los dependientes, la política establece lo siguiente:

*   **Cobertura:** El colaborador puede inscribir a su cónyuge o pareja e hijos (Sección 1.2).
*   **Cómo inscribirlos:** Debe completar el formulario en el portal de RR. HH., adjuntar el documento que acredite el vínculo (acta de matrimonio/unión o partida de nacimiento) y una copia del documento de identidad del dependiente (Sección 1.3).
*   **Plazos:** La solicitud debe enviarse dentro de los primeros 30 días desde el ingreso o desde el evento (matrimonio, nacimiento). Fuera de este plazo, se debe esperar al periodo de inscripción anual (Sección 1.3).


 ## 01. AGENTE DE CONOCIMIENTO REGLAMENTO INTERNO DE TRABAJO Y CÓDIGO DE CONDUCTA
 el primer paso es el agente va a leer la base de conocimeintos el agente debe de responder en base al documento nada mas 
librerias : **from pathlib import Path** importa la clase **Path** de la librería pathlib, que sirve para trabajar con archivos y carpetas de una forma más sencilla.

In [5]:
from pathlib import Path 
# 1. Definimos la ruta del archivo y el texto que va a contener
DOC_PATH_REGLAMENTO = "02_Reglamento_Interno.txt"
POLITICA_reglamento = """PATITO S.A.
REGLAMENTO INTERNO DE TRABAJO Y CÓDIGO DE CONDUCTA
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Políticas Internas

1. JORNADA LABORAL
Jornada de 40 horas semanales. Horario estándar de 8:00 a 17:00 con una hora de almuerzo,
salvo acuerdos de horario flexible o trabajo híbrido.

2. VACACIONES
- Cada colaborador tiene derecho a 15 días hábiles de vacaciones por año cumplido.
- Las vacaciones se solicitan a través del portal de RR. HH. con al menos 15 días de
  anticipación y deben ser aprobadas por el jefe directo.
- Pueden tomarse de forma fraccionada según acuerdo con el área.
- Los días no usados se rigen por la política de acumulación (máximo de un periodo).

3. PERMISOS
- Permisos remunerados: por matrimonio, nacimiento, fallecimiento de familiar directo, según
  la ley y la política interna.
- Permiso no remunerado: se solicita por escrito a través del portal de RR. HH., indicando el
  motivo y el periodo; requiere aprobación del jefe directo y de RR. HH. El tiempo no
  remunerado no genera remuneración durante su duración.

4. CÓDIGO DE CONDUCTA
Respeto, no discriminación, ambiente libre de acoso, cuidado de los recursos de la empresa y
confidencialidad de la información.

5. FALTAS Y SANCIONES
Las faltas se clasifican en leves, graves y muy graves, con medidas que van desde la
amonestación hasta la terminación, según la gravedad y el debido proceso."""

# 2. Instanciamos el objeto Path con la ruta del archivo
doc_reglamento_obj = Path(DOC_PATH_REGLAMENTO)

# 3. GUARDADO DEL DOCUMENTO: Si el archivo no existe en la carpeta, se crea y se escribe el texto
if not doc_reglamento_obj.exists():
    doc_reglamento_obj.write_text(POLITICA_reglamento, encoding="utf-8")
    print(f"Documento {DOC_PATH_REGLAMENTO} creado.")
else:
    print(f"El archivo {DOC_PATH_REGLAMENTO} ya existe. ")

# 4. LECTURA DEL ARCHIVO: Se lee el texto del archivo físico hacia la variable
POLITICA_reglamento = doc_reglamento_obj.read_text(encoding="utf-8")

# 5. IMPRESIÓN DE MÉTRICAS: Total de caracteres y previsualización de los primeros 400
print(f"Caracteres totales: {len(POLITICA_reglamento)}")
print(" " * 60)
print(POLITICA_reglamento[:400])



El archivo 02_Reglamento_Interno.txt ya existe. 
Caracteres totales: 1441
                                                            
PATITO S.A.
REGLAMENTO INTERNO DE TRABAJO Y CÓDIGO DE CONDUCTA
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Políticas Internas

1. JORNADA LABORAL
Jornada de 40 horas semanales. Horario estándar de 8:00 a 17:00 con una hora de almuerzo,
salvo acuerdos de horario flexible o trabajo híbrido.

2. VACACIONES
- Cada colaborador tiene derecho a 15 días h


## 02.Chunking + embeding + chroma 
**chunking:** El bloque de código que utiliza se utiliza para segmentar la informacion y se lo divide por temas **embeddings:** Esta sección convierte los fragmentos de texto plano en vectores matemáticos **chromadb:** Esta seccion permite guardar los vectores matematicos en una base de datos

In [6]:
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))
    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin].strip()
        chunks.append(parte)
    return chunks

# 1. Obtenemos los chunks del manual de reglamento interno
chunks_reglamento = chunkear_por_tema(POLITICA_reglamento)

# 2. Documentos auxiliares 
chunks_adicionales = []

# Unimos los documentos procesados
chunks_totales_reglamento = chunks_reglamento + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks_totales_reglamento)}")
print("=" * 60)

for i, c in enumerate(chunks_totales_reglamento):
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")


# 4. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_reglamento = [
    {"seccion": i + 1, "fuente": DOC_PATH_REGLAMENTO} 
    for i in range(len(chunks_totales_reglamento))
]

# 5. Inicializamos el vectorstore en Chroma

vectorstore_reglamento = Chroma.from_texts(
    texts=chunks_totales_reglamento,
    embedding=embeddings,            
    metadatas=metadatos_reglamento,
    collection_name="reglamento_interno"
)

# 6. Configuramos retriever para extraer los 2 mejores resultados (k=2)
retriever_reglamento = vectorstore_reglamento.as_retriever(search_kwargs={"k": 2})

print("\nBase de conocimiento embebida en Chroma con embeddings de Gemini.")


Total de chunks creados: 5
Chunk 1: 1. JORNADA LABORAL Jornada de 40 horas semanales. Horario estándar de ...
Chunk 2: 2. VACACIONES - Cada colaborador tiene derecho a 15 días hábiles de va...
Chunk 3: 3. PERMISOS - Permisos remunerados: por matrimonio, nacimiento, fallec...
Chunk 4: 4. CÓDIGO DE CONDUCTA Respeto, no discriminación, ambiente libre de ac...
Chunk 5: 5. FALTAS Y SANCIONES Las faltas se clasifican en leves, graves y muy ...

Base de conocimiento embebida en Chroma con embeddings de Gemini.


## 03. Agente de conocimiento (RACK) REGLAMENTO INTERNO DE TRABAJO Y CÓDIGO DE CONDUCTA


In [7]:
PROMPT_CONOCIMIENTO_REGLAMENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre el reglamento interno de trabajo y codigo de conducta.

Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""
def responder_POLITICA_reglamento(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento de reglamento y genera la respuesta."""
    docs = retriever_reglamento.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])  
    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO_REGLAMENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])  
    contenido = msg.content
    if isinstance(contenido, str):
        return contenido
    
    if isinstance(contenido, list):
        return "".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict) and bloque.get("type") == "text"
        )
    return str(contenido)
# Prueba
print(responder_POLITICA_reglamento("¿tengo vacaciones ?"))

Sí, cada colaborador tiene derecho a 15 días hábiles de vacaciones por año cumplido, según la sección 2. VACACIONES.


## 01. Agente de conocimiento 3  Reclutamiento e Onboarding

In [8]:
# 1.el primer paso es importar librerias 
from pathlib import Path 

# 2. creamos las variables de documentos y politicas y ponemos su ruta ademas si no esta el doc cargado , lo escribimos 
DOC_PATH_RECLUTAMIENTO = "03_Reclutamiento_Onboarding.txt"
RECLUTAMIENTO = """PATITO S.A.
GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Reclutamiento y Onboarding

1. PROCESO DE SELECCIÓN
Requisición del área -> publicación de la vacante -> revisión de hojas de vida -> entrevistas
(RR. HH. y área solicitante) -> evaluación técnica -> oferta -> contratación.

2. PROGRAMA DE REFERIDOS
- Cualquier colaborador puede referir candidatos para vacantes abiertas a través del portal
  de RR. HH.
- Si el referido es contratado y supera el periodo de prueba (90 días), el colaborador que lo
  refirió recibe un bono de referido.
- No aplica para posiciones de dirección ni para familiares directos del referente
  (para evitar conflicto de interés).

3. ONBOARDING (INDUCCIÓN DE NUEVOS INGRESOS)
Pasos del proceso de onboarding:
3.1 Antes del primer día: TI prepara los accesos y el equipo; RR. HH. envía la bienvenida.
3.2 Primer día: bienvenida, entrega de equipo, firma de documentos y recorrido por la empresa.
3.3 Primera semana: inducción a la cultura, políticas internas y herramientas; asignación de
    un "padrino" o mentor.
3.4 Plan 30-60-90 días: objetivos y seguimiento del nuevo colaborador con su jefe.
3.5 Evaluación de periodo de prueba a los 90 días.

4. DOCUMENTOS DE INGRESO
Identificación, datos bancarios, formularios de beneficios y contrato firmado."""

# Instanciar el objeto Path con el nuevo nombre del archivo
doc_reclutamiento_obj = Path(DOC_PATH_RECLUTAMIENTO)

# 3. Verificar si el documento existe o crearlo si no está presente
if not doc_reclutamiento_obj.exists():
    doc_reclutamiento_obj.write_text(RECLUTAMIENTO, encoding="utf-8")
    print("docpack creado")
else:
    print("El archivo ya existe. Procediendo a la lectura...")

# 4. Leer el texto del archivo utilizando .read_text() y guardarlo en la variable politica
reclutamiento = doc_reclutamiento_obj.read_text(encoding="utf-8")

# 5. Impresión de métricas requeridas e inspección de los primeros 400 caracteres
print(f"Caracteres totales: {len(reclutamiento)}")
print(" " * 60)
print(reclutamiento[:400])

El archivo ya existe. Procediendo a la lectura...
Caracteres totales: 1385
                                                            
PATITO S.A.
GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Reclutamiento y Onboarding

1. PROCESO DE SELECCIÓN
Requisición del área -> publicación de la vacante -> revisión de hojas de vida -> entrevistas
(RR. HH. y área solicitante) -> evaluación técnica -> oferta -> contratación.

2. PROGRAMA DE REFERID


## 02.chunkings +  Embeddings + ChromaDB 
**chunking:** El bloque de código que utiliza se utiliza para segmentar la informacion y se lo divide por temas **embeddings:** Esta sección convierte los fragmentos de texto plano en vectores matemáticos **chromadb:** Esta seccion permite guardar los vectores matematicos en una base de datos

In [9]:
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))
    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin].strip()
        chunks.append(parte)
    return chunks

# 1. Obtenemos los chunks del manual de reglamento interno
chunks_reclutamiento = chunkear_por_tema(RECLUTAMIENTO)

# 2. Documentos auxiliares (Vacío por ahora)
chunks_adicionales = []

# Unimos los documentos procesados
chunks_totales_reclutamiento = chunks_reclutamiento + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks_totales_reclutamiento)}")
print("=" * 60)

for i, c in enumerate(chunks_totales_reclutamiento):
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")

# 4. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_reclutamiento = [
    {"seccion": i + 1, "fuente": DOC_PATH_RECLUTAMIENTO} 
    for i in range(len(chunks_totales_reclutamiento))
]

# 5. Inicializamos el vectorstore en Chroma
vectorstore_reclutamiento = Chroma.from_texts(
    texts=chunks_totales_reclutamiento,
    embedding=embeddings,            
    metadatas=metadatos_reclutamiento,
    collection_name="reclutamiento_interno"
)

# 6. utilizamos retriever para obtener mejor resultados 
retriever_reclutamiento= vectorstore_reclutamiento.as_retriever(search_kwargs={"k": 2})

print("\nBase de conocimiento embebida en Chroma con embeddings de Gemini.")

Total de chunks creados: 4
Chunk 1: 1. PROCESO DE SELECCIÓN Requisición del área -> publicación de la vaca...
Chunk 2: 2. PROGRAMA DE REFERIDOS - Cualquier colaborador puede referir candida...
Chunk 3: 3. ONBOARDING (INDUCCIÓN DE NUEVOS INGRESOS) Pasos del proceso de onbo...
Chunk 4: 4. DOCUMENTOS DE INGRESO Identificación, datos bancarios, formularios ...

Base de conocimiento embebida en Chroma con embeddings de Gemini.


## 03 Agente de conocimiento (rack) GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING

In [10]:
PROMPT_CONOCIMIENTO_REGLAMENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING
Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""

def responder_RECLUTAMIENTO(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento de reglamento y genera la respuesta."""
    docs = retriever_reclutamiento.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])

    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO_REGLAMENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return msg.content
# Prueba
print(responder_RECLUTAMIENTO("¿ cual es el proceso de onbording?"))

[{'type': 'text', 'text': 'El proceso de onboarding en Patito S.A. (Sección 3) consta de los siguientes pasos:\n\n*   **3.1 Antes del primer día:** TI prepara accesos y equipo; RR. HH. envía la bienvenida.\n*   **3.2 Primer día:** Bienvenida, entrega de equipo, firma de documentos y recorrido por la empresa.\n*   **3.3 Primera semana:** Inducción a la cultura, políticas internas y herramientas; asignación de un "padrino" o mentor.\n*   **3.4 Plan 30-60-90 días:** Objetivos y seguimiento del nuevo colaborador con su jefe.\n*   **3.5 Evaluación:** Periodo de prueba a los 90 días.', 'extras': {'signature': 'EjQKMgERTTIPu+GMBEpWbkibfOOSGAIlR66I7sjmQ3My1Hrr2Lha9+QYk9Toe5Owg1HLz8qZ'}}]


## Agente  — Multimodal de imagen
Gemini es **multimodal**: puede recibir una imagen y leerla.

In [11]:
from PIL import Image, ImageDraw
import base64
def crear_formulario_demo(ruta="formulario_dependiente.png"):
    """Genera una imagen simple de formulario para probar el agente multimodal."""
    img = Image.new("RGB", (500, 400), "white")
    d = ImageDraw.Draw(img)
    lineas = [
        "PATITO S.A. - RECURSOS HUMANOS",
        "FORMULARIO DE INSCRIPCION DE DEPENDIENTE",
        "----------------------------------------",
        "Nombre del colaborador: Juan Perez",
        "Nombre del dependiente: Maria Perez",
        "Vinculo: Conyuge",
        "Fecha de nacimiento dependiente: ",      
        "Documento de respaldo adjunto: NO",      
        "----------------------------------------",
        "Fecha de solicitud: 2026-07-20",
        "Firma: ________________",
    ]
    y = 20
    for ln in lineas:
        d.text((20, y), ln, fill="black")
        y += 25
    img.save(ruta)
    return ruta
ruta_formulario = crear_formulario_demo()
print("formulario creado:", ruta_formulario)

formulario creado: formulario_dependiente.png


In [12]:
from langchain_core.messages import HumanMessage

def analizar_formulario(ruta_imagen: str) -> str:
    """Agente multimodal: envía la imagen a Gemini (vision) y valida/extrae los datos del formulario."""
    try:
        with open(ruta_imagen, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
    except FileNotFoundError:
        return f"No se encontró la imagen '{ruta_imagen}'."

    prompt = (
        "Analiza esta imagen de un formulario de RR.HH. de Patito S.A. "
        "Extrae y devuelve en líneas separadas: nombre del colaborador, nombre del dependiente, "
        "vínculo, fecha de nacimiento del dependiente, si tiene documento de respaldo adjunto (SI/NO), "
        "y fecha de solicitud. "
        "Si algún dato no aparece o está vacío, escribe 'no visible' o 'falta'. "
        "Al final, indica explícitamente si el formulario está COMPLETO o INCOMPLETO, "
        "y en caso de estar incompleto, lista qué datos faltan."
    )
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": f"data:image/png;base64,{b64}"},
    ])
    return llm.invoke([msg]).content

# Prueba
print(analizar_formulario("formulario_dependiente.png"))

[{'type': 'text', 'text': 'Aquí tienes la información extraída del formulario:\n\n**Nombre del colaborador:** Juan Perez\n**Nombre del dependiente:** Maria Perez\n**Vínculo:** Conyuge\n**Fecha de nacimiento del dependiente:** falta\n**Documento de respaldo adjunto:** NO\n**Fecha de solicitud:** 2026-07-20\n\n---\n**Estado del formulario:** INCOMPLETO\n\n**Datos faltantes:**\n* Fecha de nacimiento del dependiente\n* Firma del colaborador', 'extras': {'signature': 'EjQKMgERTTIP30LOwSX55BpkaNmz5XjcmBmP6jvLHjggnnzcCMPzJRU46lgeRSA711LFGXUa'}}]


## Agente de accion de registro de solicitud

In [13]:

from pathlib import Path
from datetime import datetime
from langchain.tools import tool

REGISTRO_PATH = "registro_solicitudes_rrhh.txt"

CAMPOS_VACACIONES = ["nombre_solicitante", "fecha_inicio", "fecha_fin", "dias", "jefe_aprueba"]
CAMPOS_DEPENDIENTE = ["nombre_solicitante", "nombre_dependiente", "vinculo", "documento_respaldo"]
DIAS_ANTICIPACION_MINIMOS = 15

def _siguiente_id(prefijo: str) -> str:
    if not Path(REGISTRO_PATH).exists():
        return f"{prefijo}-0001"
    n = sum(1 for l in open(REGISTRO_PATH, encoding="utf-8") if l.strip().startswith(prefijo))
    return f"{prefijo}-{n + 1:04d}"


@tool
def registrar_solicitud_rrhh(tipo: str = "", nombre_solicitante: str = "",
                              fecha_inicio: str = "", fecha_fin: str = "", dias: int = 0,
                              jefe_aprueba: str = "", nombre_dependiente: str = "",
                              vinculo: str = "", documento_respaldo: str = "",
                              confirmar: bool = False) -> str:
    """Registra una solicitud de RR.HH. en un archivo de texto. El parametro 'tipo' debe ser
    'vacaciones' o 'dependiente'.
    Si tipo='vacaciones', requiere TODOS estos datos: nombre_solicitante, fecha_inicio (YYYY-MM-DD),
    fecha_fin (YYYY-MM-DD), dias (numero de dias) y jefe_aprueba. La fecha_inicio debe tener al
    menos 15 dias de anticipacion respecto a hoy.
    Si tipo='dependiente', requiere TODOS estos datos: nombre_solicitante, nombre_dependiente,
    vinculo (ej. conyuge, hijo) y documento_respaldo.
    Si falta algun dato obligatorio o no cumple la anticipacion, NO registra y devuelve que datos
    faltan o que corregir. Solo escribe en el archivo cuando confirmar=True; si confirmar=False,
    devuelve un resumen pidiendo confirmacion explicita antes de registrar."""

    tipo = tipo.strip().lower()

    if tipo == "vacaciones":
        datos = {"nombre_solicitante": nombre_solicitante, "fecha_inicio": fecha_inicio,
                  "fecha_fin": fecha_fin, "dias": dias, "jefe_aprueba": jefe_aprueba}

        # --- SISTEMA DE CONTROL: validar campos obligatorios ---
        faltantes = [k for k in CAMPOS_VACACIONES
                     if not str(datos[k]).strip() or (k == "dias" and int(dias) <= 0)]
        if faltantes:
            return "No se registro la solicitud. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

        #  validar anticipacion minima (15 dias) ---
        try:
            fecha_inicio_dt = datetime.strptime(fecha_inicio, "%Y-%m-%d")
        except ValueError:
            return f"No se registro la solicitud: fecha_inicio '{fecha_inicio}' no tiene formato valido (YYYY-MM-DD)."

        dias_restantes = (fecha_inicio_dt - datetime.now()).days
        if dias_restantes < DIAS_ANTICIPACION_MINIMOS:
            return (f"No se registro la solicitud: se requieren al menos {DIAS_ANTICIPACION_MINIMOS} "
                    f"dias de anticipacion (faltan {DIAS_ANTICIPACION_MINIMOS - dias_restantes}).")

        firma = f"VAC|{nombre_solicitante}|{fecha_inicio}|{fecha_fin}|{dias}|{jefe_aprueba}"
        resumen = (f"{nombre_solicitante}, del {fecha_inicio} al {fecha_fin} ({dias} dias), "
                   f"aprobado por {jefe_aprueba}")
        linea_datos = (f"{nombre_solicitante} | {fecha_inicio} a {fecha_fin} | {dias} dias | "
                       f"Aprueba: {jefe_aprueba}")
        prefijo_id = "VAC"

    elif tipo == "dependiente":
        datos = {"nombre_solicitante": nombre_solicitante, "nombre_dependiente": nombre_dependiente,
                  "vinculo": vinculo, "documento_respaldo": documento_respaldo}

        # validar campos obligatorios 
        faltantes = [k for k in CAMPOS_DEPENDIENTE if not str(datos[k]).strip()]
        if faltantes:
            return "No se registro la solicitud. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

        firma = f"DEP|{nombre_solicitante}|{nombre_dependiente}|{vinculo}|{documento_respaldo}"
        resumen = (f"dependiente {nombre_dependiente} ({vinculo}) de {nombre_solicitante}, "
                   f"con respaldo: {documento_respaldo}")
        linea_datos = (f"Solicitante: {nombre_solicitante} | Dependiente: {nombre_dependiente} "
                       f"({vinculo}) | Respaldo: {documento_respaldo}")
        prefijo_id = "DEP"

    else:
        return "Tipo de solicitud no reconocido. Usa 'vacaciones' o 'dependiente'."

    #  evitar duplicados 
    if Path(REGISTRO_PATH).exists():
        with open(REGISTRO_PATH, encoding="utf-8") as f:
            if any(firma in l for l in f):
                return "Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar."

    #  pedir confirmacion antes de escribir 
    if not confirmar:
        return f"Datos completos y validos: {resumen}. ¿Confirmas el registro? Vuelve a invocar con confirmar=True para finalizar."

    rid = _siguiente_id(prefijo_id)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    linea = f"{rid} | {ts} | {linea_datos} | firma:{firma}"

    try:
        with open(REGISTRO_PATH, "a", encoding="utf-8") as f:
            f.write(linea + "\n")
    except Exception as e:
        return f"Error al registrar: {e}"

    return f"Solicitud registrada con ID {rid}.  ->  {linea}"

if __name__ == "__main__":
    # Prueba 1: faltan datos -> el control lo impide
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20"}))

    # Prueba 2: datos completos pero SIN confirmar -> pide confirmacion
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez"}))

    # Prueba 3: datos completos y CONFIRMANDO -> registra
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez", "confirmar": True}))

    # Prueba 4: mismo registro otra vez -> detecta duplicado
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez", "confirmar": True}))

    # Prueba 5: dependiente incompleto
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "dependiente", "nombre_solicitante": "Juan Perez", "nombre_dependiente": "Ana Perez"}))

    # Prueba 6: dependiente completo y confirmado
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "dependiente", "nombre_solicitante": "Juan Perez", "nombre_dependiente": "Ana Perez",
        "vinculo": "conyuge", "documento_respaldo": "cedula.pdf", "confirmar": True}))

No se registro la solicitud. Faltan datos obligatorios: fecha_fin, dias, jefe_aprueba.
Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar.
Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar.
Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar.
No se registro la solicitud. Faltan datos obligatorios: vinculo, documento_respaldo.
Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar.


## orquestador 

In [14]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import uuid

@tool
def consultar_beneficios(pregunta: str) -> str:
    """Responde preguntas sobre el Manual de Beneficios y Compensaciones: seguro medico,
    dependientes, bonos, otros beneficios y estructura de compensacion. Usa la base de
    conocimiento embebida del Agente de Beneficios."""
    return responder_politica(pregunta)


@tool
def consultar_politicas_internas(pregunta: str) -> str:
    """Responde preguntas sobre el Reglamento Interno de Trabajo y Codigo de Conducta:
    jornada laboral, vacaciones, permisos, codigo de conducta y faltas/sanciones. Usa la
    base de conocimiento embebida del Agente de Politicas Internas."""
    return responder_POLITICA_reglamento(pregunta)


@tool
def consultar_reclutamiento(pregunta: str) -> str:
    """Responde preguntas sobre el proceso de seleccion, el programa de referidos y el
    onboarding de nuevos colaboradores. Usa la base de conocimiento embebida del Agente
    de Reclutamiento y Onboarding."""
    return responder_RECLUTAMIENTO(pregunta)


@tool
def analizar_formulario_tool(ruta_imagen: str) -> str:
    """Agente multimodal: analiza la imagen de un formulario de RR.HH. (por ejemplo, el
    formulario de inscripcion de dependiente) y extrae sus datos, indicando si esta
    completo o que informacion falta. Recibe la RUTA del archivo de imagen (ej. 'formulario_dependiente.png')."""
    return analizar_formulario(ruta_imagen)


tools_orquestador = [
    consultar_beneficios,
    consultar_politicas_internas,
    consultar_reclutamiento,
    analizar_formulario_tool,
    registrar_solicitud_rrhh,
]


# Prompt del orquestador

SYSTEM_PROMPT = """Eres el orquestador de la Mesa de Ayuda IA de Recursos Humanos de Patito S.A.
Coordinas cinco capacidades (tools). NUNCA respondas de memoria: siempre usa la tool
correspondiente para obtener la informacion antes de responder.

- consultar_beneficios: preguntas sobre seguro medico, dependientes, bonos y compensacion.
- consultar_politicas_internas: preguntas sobre vacaciones, permisos, jornada laboral,
  codigo de conducta y sanciones.
- consultar_reclutamiento: preguntas sobre proceso de seleccion, programa de referidos
  y onboarding.
- analizar_formulario_tool: cuando el usuario mencione o adjunte la RUTA de una imagen
  de un formulario (ej. formulario_dependiente.png).
- registrar_solicitud_rrhh: para REGISTRAR una solicitud de vacaciones o de inscripcion
  de dependiente (tipo='vacaciones' o tipo='dependiente').

Reglas de ruteo:
- Si la pregunta toca mas de un tema (ej. vacaciones Y beneficios), DEBES invocar todas
  las tools de conocimiento relevantes y consolidar ambas respuestas en una sola, clara
  y ordenada por tema.
- Si el usuario da la ruta de una imagen, usa analizar_formulario_tool primero. Si de esa
  imagen surge un registro pendiente (ej. datos de un dependiente), puedes complementarlo
  con consultar_beneficios antes de responder.
- Si el usuario pide registrar/guardar una solicitud, usa registrar_solicitud_rrhh.
  Necesitas, segun el tipo:
  - vacaciones: nombre_solicitante, fecha_inicio, fecha_fin, dias, jefe_aprueba
    (fecha_inicio con al menos 15 dias de anticipacion).
  - dependiente: nombre_solicitante, nombre_dependiente, vinculo, documento_respaldo.
  Si falta algun dato, PIDESELO al usuario y espera su respuesta; nunca registres con
  datos incompletos ni sin que el usuario confirme explicitamente (confirmar=True solo
  despues de que el usuario diga que si).
- Si ninguna tool de conocimiento devuelve informacion relevante, responde exactamente:
  "No encontre informacion suficiente en la base documental proporcionada." No inventes
  datos que no esten en el contexto recuperado.
- Al final de cada respuesta, agrega una linea "Agentes utilizados: ..." indicando que
  tool(s) invocaste, para dar trazabilidad."""


# Memoria: permite conversaciones multi-turno 

memoria = InMemorySaver()
orquestador = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria,
)

print("Tools registradas en el orquestador:")
for t in tools_orquestador:
    print("  -", t.name)


def _imprimir_pasos(resultado):
    """Muestra que tools se invocaron y su resultado (trazabilidad)."""
    for m in resultado["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print(f"[TOOL] {tc['name']}({tc['args']})")
        if m.__class__.__name__ == "ToolMessage":
            print(f"[RESPONSE] {str(m.content)[:300]}\n")


def extraer_texto(content):
    """Gemini a veces devuelve el content como una LISTA de bloques
    (texto + firmas de 'thinking'). Esta funcion devuelve solo el texto plano."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        partes = []
        for b in content:
            if isinstance(b, dict):
                partes.append(b.get("text", ""))
            elif isinstance(b, str):
                partes.append(b)
        return "".join(partes).strip()
    return str(content)


def consultar(pregunta: str, thread_id: str = None):
    """Invoca al orquestador (una consulta suelta) e imprime tools + respuesta final."""
    thread_id = thread_id or f"demo-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}
    print(f">>> Usuario: {pregunta}\n")
    resultado = orquestador.invoke({"messages": [{"role": "user", "content": pregunta}]}, config)
    _imprimir_pasos(resultado)
    print("=== Respuesta final ===")
    print(extraer_texto(resultado["messages"][-1].content))
    return resultado


Tools registradas en el orquestador:
  - consultar_beneficios
  - consultar_politicas_internas
  - consultar_reclutamiento
  - analizar_formulario_tool
  - registrar_solicitud_rrhh


In [15]:
# prueba 1 : agente de conocimeinto simple 
if __name__ == "__main__":
  consultar("¿Cuántos días de vacaciones me corresponden al año?")

>>> Usuario: ¿Cuántos días de vacaciones me corresponden al año?

[TOOL] consultar_politicas_internas({'pregunta': '¿Cuántos días de vacaciones me corresponden al año?'})
[RESPONSE] De acuerdo con la sección 2, cada colaborador tiene derecho a 15 días hábiles de vacaciones por año cumplido.

=== Respuesta final ===
De acuerdo con el Reglamento Interno de Trabajo, cada colaborador tiene derecho a 15 días hábiles de vacaciones por cada año cumplido de servicio.

Agentes utilizados: consultar_politicas_internas


In [16]:
# Prueba 2: consulta mixta (vacaciones + beneficios) 
consultar(
        "Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. "
        "¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir "
        "a un dependiente en el beneficio?"
    )


>>> Usuario: Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. ¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir a un dependiente en el beneficio?

[TOOL] consultar_politicas_internas({'pregunta': '¿Cuántos días de vacaciones corresponden y cómo se solicitan?'})
[TOOL] consultar_beneficios({'pregunta': '¿Qué necesito para inscribir a mi pareja como dependiente en el seguro médico?'})
[RESPONSE] Corresponden 15 días hábiles de vacaciones por año cumplido. Se solicitan a través del portal de RR. HH. con al menos 15 días de anticipación y requieren la aprobación del jefe directo (Sección 2).

[RESPONSE] Para inscribir a tu pareja como dependiente, debes realizar lo siguiente según la sección 1.3:

*   Completar el formulario de inscripción de dependientes en el portal de RR. HH.
*   Adjuntar el documento que acredite el vínculo (acta de matrimonio o unión).
*   Adjuntar copia del documento de ident

=== Respuesta final ===
Para g

{'messages': [HumanMessage(content='Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. ¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir a un dependiente en el beneficio?', additional_kwargs={}, response_metadata={}, id='39222c61-18b4-48b2-9831-ccd48469d0d0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_beneficios', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 necesito para inscribir a mi pareja como dependiente en el seguro m\\u00e9dico?"}'}, '__gemini_function_call_thought_signatures__': {'976cmpgu': 'EjQKMgERTTIPXN2Xc58du0p78PMBPYtkaS3z5SmPEMc8WiwWXF05F6J8g0T5YKou/uqU3HiY'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9d2e-7c1c-7613-95be-2af555a8875d-0', tool_calls=[{'name': 'consultar_politicas_internas', 'args': {'pregunta': '¿Cuántos días de vacaciones corresponden y cómo se 

In [ ]:
# Prueba 3: multimodal
consultar("Adjunto el formulario en formulario_dependiente.png: ¿está completo y qué datos faltan?")


In [ ]:
 # Prueba 4: registrar con datos incompletos -> el sistema de control debe pedir lo que falta
consultar("Registra una solicitud de vacaciones para Juan Perez.")



In [ ]:
# Prueba 5: fuera de alcance -> debe admitir que no tiene informacion
consultar("¿Cuál es el precio de las acciones de Patito S.A. en la bolsa?")


## Agente Interactivo 

## Fronend

In [ ]:
pip install streamlit python-dotenv


In [20]:
"""
app.py — Interfaz web (Streamlit) para la Mesa de Ayuda IA de Patito S.A.
Ejecutar con: streamlit run app.py
"""

import uuid
import streamlit as st

st.set_page_config(page_title="Mesa de Ayuda IA — Patito S.A.", page_icon="🦆", layout="centered")

# ---------------------------------------------------------
# Cargar el backend (orquestador) con manejo de errores
# ---------------------------------------------------------
try:
    from backend import consultar
    backend_disponible = True
    error_backend = None
except Exception as e:
    backend_disponible = False
    error_backend = str(e)

st.title("🦆 Mesa de Ayuda IA — Recursos Humanos")
st.caption("Patito S.A. · Beneficios · Políticas Internas · Reclutamiento · Formularios · Registro de solicitudes")

if not backend_disponible:
    st.error(
        "No se pudo cargar el backend (orquestador). Revisa que backend.py esté completo "
        "y que la GOOGLE_API_KEY esté configurada en tu archivo .env.\n\n"
        f"Detalle del error: {error_backend}"
    )
    st.stop()

# ---------------------------------------------------------
# Memoria de la conversación (thread_id fijo por sesión de navegador)
# ---------------------------------------------------------
if "thread_id" not in st.session_state:
    st.session_state.thread_id = f"streamlit-{uuid.uuid4().hex[:8]}"

if "historial" not in st.session_state:
    st.session_state.historial = []  # lista de (pregunta, respuesta, agentes_usados)

# ---------------------------------------------------------
# Barra lateral: subir imagen (para el agente multimodal)
# ---------------------------------------------------------
with st.sidebar:
    st.header("📎 Adjuntar formulario (opcional)")
    st.caption("Para el agente multimodal: sube una imagen y su ruta se agregará a tu pregunta.")
    imagen = st.file_uploader("Formulario / comprobante", type=["png", "jpg", "jpeg"])
    ruta_imagen_actual = None
    if imagen is not None:
        ruta_imagen_actual = f"_subida_{imagen.name}"
        with open(ruta_imagen_actual, "wb") as f:
            f.write(imagen.getbuffer())
        st.image(imagen, caption="Vista previa", use_container_width=True)
        st.success(f"Imagen guardada como: {ruta_imagen_actual}")

    st.divider()
    if st.button("🗑️ Reiniciar conversación"):
        st.session_state.historial = []
        st.session_state.thread_id = f"streamlit-{uuid.uuid4().hex[:8]}"
        st.rerun()

# ---------------------------------------------------------
# Mostrar historial de la conversación
# ---------------------------------------------------------
for pregunta, respuesta, agentes in st.session_state.historial:
    with st.chat_message("user"):
        st.write(pregunta)
    with st.chat_message("assistant"):
        st.write(respuesta)
        if agentes:
            st.caption(f"🔎 Agentes usados: {', '.join(agentes)}")

# ---------------------------------------------------------
# Entrada de la pregunta (estilo chat)
# ---------------------------------------------------------
pregunta_usuario = st.chat_input("Escribe tu pregunta para RR.HH...")

if pregunta_usuario:
    # Si hay imagen subida, se agrega la ruta a la pregunta para que el orquestador la detecte
    pregunta_final = pregunta_usuario
    if ruta_imagen_actual:
        pregunta_final = f"{pregunta_usuario} (imagen adjunta en la ruta: {ruta_imagen_actual})"

    with st.chat_message("user"):
        st.write(pregunta_usuario)

    with st.chat_message("assistant"):
        with st.spinner("Consultando a los agentes..."):
            try:
                config = {"configurable": {"thread_id": st.session_state.thread_id}}
                resultado = consultar(pregunta_final, thread_id=st.session_state.thread_id)

                # Extraer texto final y tools usadas para trazabilidad
                mensajes = resultado["messages"]
                agentes_usados = []
                for m in mensajes:
                    for tc in (getattr(m, "tool_calls", None) or []):
                        agentes_usados.append(tc["name"])

                contenido = mensajes[-1].content
                if isinstance(contenido, list):
                    texto_final = "".join(
                        b.get("text", "") for b in contenido if isinstance(b, dict)
                    ).strip()
                else:
                    texto_final = str(contenido)

            except Exception as e:
                texto_final = f"Ocurrió un error al consultar los agentes: {e}"
                agentes_usados = []

        st.write(texto_final)
        if agentes_usados:
            st.caption(f"🔎 Agentes usados: {', '.join(agentes_usados)}")

    st.session_state.historial.append((pregunta_usuario, texto_final, agentes_usados))

2026-07-26 02:00:14.412 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 02:00:14.428 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 02:00:15.444 
  command:

    streamlit run C:\Users\GEO\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-26 02:00:15.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 02:00:15.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 02:00:15.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 02:00:15.476 Thread 'MainThread': missing ScriptRunContext! This warn

## backend 

In [19]:
 
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv()  
 
if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError(
        "Falta GOOGLE_API_KEY. Crea un archivo .env con GOOGLE_API_KEY= "
        "(usa .env.example como plantilla)."
    )
 
MODELO_LLM = "gemini-flash-latest"
MODELO_EMBEDDING = "models/gemini-embedding-001"
llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)
#-- los agentes racks 
from pathlib import Path 

# 2. creamos las variables de documentos y politicas y ponemos su ruta ademas si no esta el doc cargado , lo escribimos 
DOC_PATH = "01_Beneficios_Compensaciones.txt"
POLITICA = """PATITO S.A.
MANUAL DE BENEFICIOS Y COMPENSACIONES
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Beneficios y Compensaciones

1. SEGURO MÉDICO CORPORATIVO
1.1 Cobertura: consultas médicas, hospitalización, emergencias, exámenes de laboratorio y
    medicamentos según el plan. Incluye atención ambulatoria y cobertura dental básica.
1.2 Dependientes: el colaborador puede inscribir a cónyuge o pareja e hijos.
1.3 Cómo agregar un dependiente:
    - Completar el formulario de inscripción de dependientes en el portal de RR. HH.
    - Adjuntar el documento que acredite el vínculo (acta de matrimonio/unión o partida de
      nacimiento) y copia del documento de identidad del dependiente.
    - Enviar la solicitud dentro de los primeros 30 días desde el ingreso o desde el evento
      (matrimonio, nacimiento). Fuera de ese plazo, se espera al periodo de inscripción anual.

2. BONOS
- Bono por desempeño anual según evaluación.
- Bono por cumplimiento de metas del área (cuando aplique).

3. OTROS BENEFICIOS
- Día libre de cumpleaños.
- Capacitación y apoyo educativo.
- Modalidad híbrida según el puesto.

4. COMPENSACIÓN
La estructura salarial considera el rol, la banda salarial y el mercado. Las revisiones
salariales se realizan una vez al año."""

# 3.Guardar la instancia path en la variable doc_path_obj
doc_path_obj = Path(DOC_PATH)

# 4. Verificar si el documento existe o crearlo si no está presente
if not doc_path_obj.exists():
    doc_path_obj.write_text(POLITICA, encoding="utf-8")
    print("docpack creado")
else:
    print("El archivo ya existe. Procediendo a la lectura...")

# 5.Utilizar el comando .read_text() para leer el doc y guardarlo en la variable politica
politica = doc_path_obj.read_text(encoding="utf-8")

# 6. Impresión de métricas requeridas de los primeros 400 caracteres
print(f"Caracteres totales: {len(politica)}")
print(" " * 60)
print(politica[:400])

# embeding del primer agente 
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    
    # se crea la variable cabecera donde se guardara la lista de los temas buscados por segmentos 
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))

    chunks = []

    for i, m in enumerate(cabeceras):
        ini = m.start()
        # Si no es el último tema, el final es donde empieza el siguiente. Si es el último, va hasta el fin del texto.
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)

        # Extraemos la sección correspondiente y limpiamos espacios en blanco innecesarios
        parte = texto[ini:fin].strip()
        chunks.append(parte)

    return chunks

# 1. Obtenemos los chunks del manual de beneficios y compensaciones y la guardamos en la variable chunk_beneficios 

chunks_beneficios = chunkear_por_tema(politica)

# 2. en la varible chunks adicionales se guardan documentos auxiliares, si llegaran a existir 
chunks_adicionales = [] 

# Unimos los documentos procesados
chunks = chunks_beneficios + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks)}")
print("=" * 60)

for i, c in enumerate(chunks):
    # Reemplazamos los saltos de línea internos por espacios solo para la previsualización del print
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")
    
# Cada chunk se embebe con Gemini y se guarda en Chroma
# 1. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_chunks = [{"seccion": i + 1, "fuente": DOC_PATH} for i in range(len(chunks))]

# 2. Inicializamos el vectorstore en Chroma con la configuración e inserción de datos corregida
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    metadatas=metadatos_chunks,
    collection_name="beneficios_y_compensaciones"
)

# 3. Configuramos retriever para extraer los 2 mejores resultados (k=2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# rack del primer agente 
PROMPT_CONOCIMIENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre el manual de beneficios y compensaciones.
Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""
def responder_politica(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento y genera la respuesta."""
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])
    
    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    contenido = msg.content
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        return "".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict) and bloque.get("type") == "text"
        )
    return str(contenido)

#-- agente 2 

from pathlib import Path 
# 1. Definimos la ruta del archivo y el texto que va a contener
DOC_PATH_REGLAMENTO = "02_Reglamento_Interno.txt"
POLITICA_reglamento = """PATITO S.A.
REGLAMENTO INTERNO DE TRABAJO Y CÓDIGO DE CONDUCTA
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Políticas Internas

1. JORNADA LABORAL
Jornada de 40 horas semanales. Horario estándar de 8:00 a 17:00 con una hora de almuerzo,
salvo acuerdos de horario flexible o trabajo híbrido.

2. VACACIONES
- Cada colaborador tiene derecho a 15 días hábiles de vacaciones por año cumplido.
- Las vacaciones se solicitan a través del portal de RR. HH. con al menos 15 días de
  anticipación y deben ser aprobadas por el jefe directo.
- Pueden tomarse de forma fraccionada según acuerdo con el área.
- Los días no usados se rigen por la política de acumulación (máximo de un periodo).

3. PERMISOS
- Permisos remunerados: por matrimonio, nacimiento, fallecimiento de familiar directo, según
  la ley y la política interna.
- Permiso no remunerado: se solicita por escrito a través del portal de RR. HH., indicando el
  motivo y el periodo; requiere aprobación del jefe directo y de RR. HH. El tiempo no
  remunerado no genera remuneración durante su duración.

4. CÓDIGO DE CONDUCTA
Respeto, no discriminación, ambiente libre de acoso, cuidado de los recursos de la empresa y
confidencialidad de la información.

5. FALTAS Y SANCIONES
Las faltas se clasifican en leves, graves y muy graves, con medidas que van desde la
amonestación hasta la terminación, según la gravedad y el debido proceso."""

# 2. Instanciamos el objeto Path con la ruta del archivo
doc_reglamento_obj = Path(DOC_PATH_REGLAMENTO)

# 3. GUARDADO DEL DOCUMENTO: Si el archivo no existe en la carpeta, se crea y se escribe el texto
if not doc_reglamento_obj.exists():
    doc_reglamento_obj.write_text(POLITICA_reglamento, encoding="utf-8")
    print(f"Documento {DOC_PATH_REGLAMENTO} creado.")
else:
    print(f"El archivo {DOC_PATH_REGLAMENTO} ya existe. ")

# 4. LECTURA DEL ARCHIVO: Se lee el texto del archivo físico hacia la variable
POLITICA_reglamento = doc_reglamento_obj.read_text(encoding="utf-8")

# 5. IMPRESIÓN DE MÉTRICAS: Total de caracteres y previsualización de los primeros 400
print(f"Caracteres totales: {len(POLITICA_reglamento)}")
print(" " * 60)
print(POLITICA_reglamento[:400])

# embeding del segundo agente 
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))
    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin].strip()
        chunks.append(parte)
    return chunks

# 1. Obtenemos los chunks del manual de reglamento interno
chunks_reglamento = chunkear_por_tema(POLITICA_reglamento)

# 2. Documentos auxiliares 
chunks_adicionales = []

# Unimos los documentos procesados
chunks_totales_reglamento = chunks_reglamento + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks_totales_reglamento)}")
print("=" * 60)

for i, c in enumerate(chunks_totales_reglamento):
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")


# 4. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_reglamento = [
    {"seccion": i + 1, "fuente": DOC_PATH_REGLAMENTO} 
    for i in range(len(chunks_totales_reglamento))
]

# 5. Inicializamos el vectorstore en Chroma

vectorstore_reglamento = Chroma.from_texts(
    texts=chunks_totales_reglamento,
    embedding=embeddings,            
    metadatas=metadatos_reglamento,
    collection_name="reglamento_interno"
)

# 6. Configuramos retriever para extraer los 2 mejores resultados (k=2)
retriever_reglamento = vectorstore_reglamento.as_retriever(search_kwargs={"k": 2})

# rack del segundo agente 

PROMPT_CONOCIMIENTO_REGLAMENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre el reglamento interno de trabajo y codigo de conducta.

Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""
def responder_POLITICA_reglamento(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento de reglamento y genera la respuesta."""
    docs = retriever_reglamento.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])  
    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO_REGLAMENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])  
    contenido = msg.content
    if isinstance(contenido, str):
        return contenido
    
    if isinstance(contenido, list):
        return "".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict) and bloque.get("type") == "text"
        )
    return str(contenido)

# agnete 3 
# 1.el primer paso es importar librerias 
from pathlib import Path 

# 2. creamos las variables de documentos y politicas y ponemos su ruta ademas si no esta el doc cargado , lo escribimos 
DOC_PATH_RECLUTAMIENTO = "03_Reclutamiento_Onboarding.txt"
RECLUTAMIENTO = """PATITO S.A.
GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Reclutamiento y Onboarding

1. PROCESO DE SELECCIÓN
Requisición del área -> publicación de la vacante -> revisión de hojas de vida -> entrevistas
(RR. HH. y área solicitante) -> evaluación técnica -> oferta -> contratación.

2. PROGRAMA DE REFERIDOS
- Cualquier colaborador puede referir candidatos para vacantes abiertas a través del portal
  de RR. HH.
- Si el referido es contratado y supera el periodo de prueba (90 días), el colaborador que lo
  refirió recibe un bono de referido.
- No aplica para posiciones de dirección ni para familiares directos del referente
  (para evitar conflicto de interés).

3. ONBOARDING (INDUCCIÓN DE NUEVOS INGRESOS)
Pasos del proceso de onboarding:
3.1 Antes del primer día: TI prepara los accesos y el equipo; RR. HH. envía la bienvenida.
3.2 Primer día: bienvenida, entrega de equipo, firma de documentos y recorrido por la empresa.
3.3 Primera semana: inducción a la cultura, políticas internas y herramientas; asignación de
    un "padrino" o mentor.
3.4 Plan 30-60-90 días: objetivos y seguimiento del nuevo colaborador con su jefe.
3.5 Evaluación de periodo de prueba a los 90 días.

4. DOCUMENTOS DE INGRESO
Identificación, datos bancarios, formularios de beneficios y contrato firmado."""

# Instanciar el objeto Path con el nuevo nombre del archivo
doc_reclutamiento_obj = Path(DOC_PATH_RECLUTAMIENTO)

# 3. Verificar si el documento existe o crearlo si no está presente
if not doc_reclutamiento_obj.exists():
    doc_reclutamiento_obj.write_text(RECLUTAMIENTO, encoding="utf-8")
    print("docpack creado")
else:
    print("El archivo ya existe. Procediendo a la lectura...")

# 4. Leer el texto del archivo utilizando .read_text() y guardarlo en la variable politica
reclutamiento = doc_reclutamiento_obj.read_text(encoding="utf-8")

# 5. Impresión de métricas requeridas e inspección de los primeros 400 caracteres
print(f"Caracteres totales: {len(reclutamiento)}")
print(" " * 60)
print(reclutamiento[:400])

# embdeing del 3er agente 
import re
from langchain_community.vectorstores import Chroma

def chunkear_por_tema(texto):
    """Divide el manual en un chunk por cada tema numerado (1., 2., 3., etc.)."""
    cabeceras = list(re.finditer(r"^\d+\.\s+[A-ZÁÉÍÓÚÑ]", texto, flags=re.MULTILINE))
    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin].strip()
        chunks.append(parte)
    return chunks

# 1. Obtenemos los chunks del manual de reglamento interno
chunks_reclutamiento = chunkear_por_tema(RECLUTAMIENTO)

# 2. Documentos auxiliares (Vacío por ahora)
chunks_adicionales = []

# Unimos los documentos procesados
chunks_totales_reclutamiento = chunks_reclutamiento + chunks_adicionales

# 3. Imprimir métricas de verificación
print(f"Total de chunks creados: {len(chunks_totales_reclutamiento)}")
print("=" * 60)

for i, c in enumerate(chunks_totales_reclutamiento):
    vista_previa = c.replace('\n', ' ')
    print(f"Chunk {i+1}: {vista_previa[:70]}...")

# 4. Creamos la lista de metadatos dinámicamente para cada chunk
metadatos_reclutamiento = [
    {"seccion": i + 1, "fuente": DOC_PATH_RECLUTAMIENTO} 
    for i in range(len(chunks_totales_reclutamiento))
]

# 5. Inicializamos el vectorstore en Chroma
vectorstore_reclutamiento = Chroma.from_texts(
    texts=chunks_totales_reclutamiento,
    embedding=embeddings,            
    metadatas=metadatos_reclutamiento,
    collection_name="reclutamiento_interno"
)

# 6. utilizamos retriever para obtener mejor resultados 
retriever_reclutamiento= vectorstore_reclutamiento.as_retriever(search_kwargs={"k": 2})

# rack del 3r agente 
PROMPT_CONOCIMIENTO_RECLUTAMIENTO = """Eres el asistente de Recursos Humanos de Patito S.A. Respondes sobre GUÍA DE RECLUTAMIENTO, REFERIDOS Y ONBOARDING
Reglas estrictas:
- Responde ÚNICAMENTE con base en el CONTEXTO entregado.
- Cita el número de sección cuando sea posible.
- Si la información no está en el contexto, responde exactamente: "No tengo esa información en la política."
- Se breve y directo. No inventes datos."""

def responder_RECLUTAMIENTO(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto de la base de conocimiento de reglamento y genera la respuesta."""
    docs = retriever_reclutamiento.invoke(pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])

    msg = llm.invoke([
        {"role": "system", "content": PROMPT_CONOCIMIENTO_RECLUTAMIENTO},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return msg.content

# agente multimodal - crear formulario 
from PIL import Image, ImageDraw
import base64
def crear_formulario_demo(ruta="formulario_dependiente.png"):
    """Genera una imagen simple de formulario para probar el agente multimodal."""
    img = Image.new("RGB", (500, 400), "white")
    d = ImageDraw.Draw(img)
    lineas = [
        "PATITO S.A. - RECURSOS HUMANOS",
        "FORMULARIO DE INSCRIPCION DE DEPENDIENTE",
        "----------------------------------------",
        "Nombre del colaborador: Juan Perez",
        "Nombre del dependiente: Maria Perez",
        "Vinculo: Conyuge",
        "Fecha de nacimiento dependiente: ",      
        "Documento de respaldo adjunto: NO",      
        "----------------------------------------",
        "Fecha de solicitud: 2026-07-20",
        "Firma: ________________",
    ]
    y = 20
    for ln in lineas:
        d.text((20, y), ln, fill="black")
        y += 25
    img.save(ruta)
    return ruta
ruta_formulario = crear_formulario_demo()

# registro formulario - multimodal 
from langchain_core.messages import HumanMessage

def analizar_formulario(ruta_imagen: str) -> str:
    """Agente multimodal: envía la imagen a Gemini (vision) y valida/extrae los datos del formulario."""
    try:
        with open(ruta_imagen, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
    except FileNotFoundError:
        return f"No se encontró la imagen '{ruta_imagen}'."

    prompt = (
        "Analiza esta imagen de un formulario de RR.HH. de Patito S.A. "
        "Extrae y devuelve en líneas separadas: nombre del colaborador, nombre del dependiente, "
        "vínculo, fecha de nacimiento del dependiente, si tiene documento de respaldo adjunto (SI/NO), "
        "y fecha de solicitud. "
        "Si algún dato no aparece o está vacío, escribe 'no visible' o 'falta'. "
        "Al final, indica explícitamente si el formulario está COMPLETO o INCOMPLETO, "
        "y en caso de estar incompleto, lista qué datos faltan."
    )
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": f"data:image/png;base64,{b64}"},
    ])
    return llm.invoke([msg]).content

    #-- agente de accion 
   
from pathlib import Path
from datetime import datetime
from langchain.tools import tool

REGISTRO_PATH = "registro_solicitudes_rrhh.txt"

CAMPOS_VACACIONES = ["nombre_solicitante", "fecha_inicio", "fecha_fin", "dias", "jefe_aprueba"]
CAMPOS_DEPENDIENTE = ["nombre_solicitante", "nombre_dependiente", "vinculo", "documento_respaldo"]
DIAS_ANTICIPACION_MINIMOS = 15

def _siguiente_id(prefijo: str) -> str:
    if not Path(REGISTRO_PATH).exists():
        return f"{prefijo}-0001"
    n = sum(1 for l in open(REGISTRO_PATH, encoding="utf-8") if l.strip().startswith(prefijo))
    return f"{prefijo}-{n + 1:04d}"


@tool
def registrar_solicitud_rrhh(tipo: str = "", nombre_solicitante: str = "",
                              fecha_inicio: str = "", fecha_fin: str = "", dias: int = 0,
                              jefe_aprueba: str = "", nombre_dependiente: str = "",
                              vinculo: str = "", documento_respaldo: str = "",
                              confirmar: bool = False) -> str:
    """Registra una solicitud de RR.HH. en un archivo de texto. El parametro 'tipo' debe ser
    'vacaciones' o 'dependiente'.
    Si tipo='vacaciones', requiere TODOS estos datos: nombre_solicitante, fecha_inicio (YYYY-MM-DD),
    fecha_fin (YYYY-MM-DD), dias (numero de dias) y jefe_aprueba. La fecha_inicio debe tener al
    menos 15 dias de anticipacion respecto a hoy.
    Si tipo='dependiente', requiere TODOS estos datos: nombre_solicitante, nombre_dependiente,
    vinculo (ej. conyuge, hijo) y documento_respaldo.
    Si falta algun dato obligatorio o no cumple la anticipacion, NO registra y devuelve que datos
    faltan o que corregir. Solo escribe en el archivo cuando confirmar=True; si confirmar=False,
    devuelve un resumen pidiendo confirmacion explicita antes de registrar."""

    tipo = tipo.strip().lower()

    if tipo == "vacaciones":
        datos = {"nombre_solicitante": nombre_solicitante, "fecha_inicio": fecha_inicio,
                  "fecha_fin": fecha_fin, "dias": dias, "jefe_aprueba": jefe_aprueba}

        # --- SISTEMA DE CONTROL: validar campos obligatorios ---
        faltantes = [k for k in CAMPOS_VACACIONES
                     if not str(datos[k]).strip() or (k == "dias" and int(dias) <= 0)]
        if faltantes:
            return "No se registro la solicitud. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

        #  validar anticipacion minima (15 dias) ---
        try:
            fecha_inicio_dt = datetime.strptime(fecha_inicio, "%Y-%m-%d")
        except ValueError:
            return f"No se registro la solicitud: fecha_inicio '{fecha_inicio}' no tiene formato valido (YYYY-MM-DD)."

        dias_restantes = (fecha_inicio_dt - datetime.now()).days
        if dias_restantes < DIAS_ANTICIPACION_MINIMOS:
            return (f"No se registro la solicitud: se requieren al menos {DIAS_ANTICIPACION_MINIMOS} "
                    f"dias de anticipacion (faltan {DIAS_ANTICIPACION_MINIMOS - dias_restantes}).")

        firma = f"VAC|{nombre_solicitante}|{fecha_inicio}|{fecha_fin}|{dias}|{jefe_aprueba}"
        resumen = (f"{nombre_solicitante}, del {fecha_inicio} al {fecha_fin} ({dias} dias), "
                   f"aprobado por {jefe_aprueba}")
        linea_datos = (f"{nombre_solicitante} | {fecha_inicio} a {fecha_fin} | {dias} dias | "
                       f"Aprueba: {jefe_aprueba}")
        prefijo_id = "VAC"

    elif tipo == "dependiente":
        datos = {"nombre_solicitante": nombre_solicitante, "nombre_dependiente": nombre_dependiente,
                  "vinculo": vinculo, "documento_respaldo": documento_respaldo}

        # validar campos obligatorios 
        faltantes = [k for k in CAMPOS_DEPENDIENTE if not str(datos[k]).strip()]
        if faltantes:
            return "No se registro la solicitud. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

        firma = f"DEP|{nombre_solicitante}|{nombre_dependiente}|{vinculo}|{documento_respaldo}"
        resumen = (f"dependiente {nombre_dependiente} ({vinculo}) de {nombre_solicitante}, "
                   f"con respaldo: {documento_respaldo}")
        linea_datos = (f"Solicitante: {nombre_solicitante} | Dependiente: {nombre_dependiente} "
                       f"({vinculo}) | Respaldo: {documento_respaldo}")
        prefijo_id = "DEP"

    else:
        return "Tipo de solicitud no reconocido. Usa 'vacaciones' o 'dependiente'."

    #  evitar duplicados 
    if Path(REGISTRO_PATH).exists():
        with open(REGISTRO_PATH, encoding="utf-8") as f:
            if any(firma in l for l in f):
                return "Esta solicitud ya habia sido registrada previamente (duplicado). No se volvio a registrar."

    #  pedir confirmacion antes de escribir 
    if not confirmar:
        return f"Datos completos y validos: {resumen}. ¿Confirmas el registro? Vuelve a invocar con confirmar=True para finalizar."

    rid = _siguiente_id(prefijo_id)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    linea = f"{rid} | {ts} | {linea_datos} | firma:{firma}"

    try:
        with open(REGISTRO_PATH, "a", encoding="utf-8") as f:
            f.write(linea + "\n")
    except Exception as e:
        return f"Error al registrar: {e}"

    return f"Solicitud registrada con ID {rid}.  ->  {linea}"

if __name__ == "__main__":
    # Prueba 1: faltan datos -> el control lo impide
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20"}))

    # Prueba 2: datos completos pero SIN confirmar -> pide confirmacion
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez"}))

    # Prueba 3: datos completos y CONFIRMANDO -> registra
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez", "confirmar": True}))

    # Prueba 4: mismo registro otra vez -> detecta duplicado
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "vacaciones", "nombre_solicitante": "Juan Perez", "fecha_inicio": "2026-08-20",
        "fecha_fin": "2026-08-27", "dias": 5, "jefe_aprueba": "Maria Gomez", "confirmar": True}))

    # Prueba 5: dependiente incompleto
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "dependiente", "nombre_solicitante": "Juan Perez", "nombre_dependiente": "Ana Perez"}))

    # Prueba 6: dependiente completo y confirmado
    print(registrar_solicitud_rrhh.invoke({
        "tipo": "dependiente", "nombre_solicitante": "Juan Perez", "nombre_dependiente": "Ana Perez",
        "vinculo": "conyuge", "documento_respaldo": "cedula.pdf", "confirmar": True}))
    #-- agente orquestador 
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import uuid

@tool
def consultar_beneficios(pregunta: str) -> str:
    """Responde preguntas sobre el Manual de Beneficios y Compensaciones: seguro medico,
    dependientes, bonos, otros beneficios y estructura de compensacion. Usa la base de
    conocimiento embebida del Agente de Beneficios."""
    return responder_politica(pregunta)


@tool
def consultar_politicas_internas(pregunta: str) -> str:
    """Responde preguntas sobre el Reglamento Interno de Trabajo y Codigo de Conducta:
    jornada laboral, vacaciones, permisos, codigo de conducta y faltas/sanciones. Usa la
    base de conocimiento embebida del Agente de Politicas Internas."""
    return responder_POLITICA_reglamento(pregunta)


@tool
def consultar_reclutamiento(pregunta: str) -> str:
    """Responde preguntas sobre el proceso de seleccion, el programa de referidos y el
    onboarding de nuevos colaboradores. Usa la base de conocimiento embebida del Agente
    de Reclutamiento y Onboarding."""
    return responder_RECLUTAMIENTO(pregunta)


@tool
def analizar_formulario_tool(ruta_imagen: str) -> str:
    """Agente multimodal: analiza la imagen de un formulario de RR.HH. (por ejemplo, el
    formulario de inscripcion de dependiente) y extrae sus datos, indicando si esta
    completo o que informacion falta. Recibe la RUTA del archivo de imagen (ej. 'formulario_dependiente.png')."""
    return analizar_formulario(ruta_imagen)


tools_orquestador = [
    consultar_beneficios,
    consultar_politicas_internas,
    consultar_reclutamiento,
    analizar_formulario_tool,
    registrar_solicitud_rrhh,
]


# Prompt del orquestador

SYSTEM_PROMPT = """Eres el orquestador de la Mesa de Ayuda IA de Recursos Humanos de Patito S.A.
Coordinas cinco capacidades (tools). NUNCA respondas de memoria: siempre usa la tool
correspondiente para obtener la informacion antes de responder.

- consultar_beneficios: preguntas sobre seguro medico, dependientes, bonos y compensacion.
- consultar_politicas_internas: preguntas sobre vacaciones, permisos, jornada laboral,
  codigo de conducta y sanciones.
- consultar_reclutamiento: preguntas sobre proceso de seleccion, programa de referidos
  y onboarding.
- analizar_formulario_tool: cuando el usuario mencione o adjunte la RUTA de una imagen
  de un formulario (ej. formulario_dependiente.png).
- registrar_solicitud_rrhh: para REGISTRAR una solicitud de vacaciones o de inscripcion
  de dependiente (tipo='vacaciones' o tipo='dependiente').

Reglas de ruteo:
- Si la pregunta toca mas de un tema (ej. vacaciones Y beneficios), DEBES invocar todas
  las tools de conocimiento relevantes y consolidar ambas respuestas en una sola, clara
  y ordenada por tema.
- Si el usuario da la ruta de una imagen, usa analizar_formulario_tool primero. Si de esa
  imagen surge un registro pendiente (ej. datos de un dependiente), puedes complementarlo
  con consultar_beneficios antes de responder.
- Si el usuario pide registrar/guardar una solicitud, usa registrar_solicitud_rrhh.
  Necesitas, segun el tipo:
  - vacaciones: nombre_solicitante, fecha_inicio, fecha_fin, dias, jefe_aprueba
    (fecha_inicio con al menos 15 dias de anticipacion).
  - dependiente: nombre_solicitante, nombre_dependiente, vinculo, documento_respaldo.
  Si falta algun dato, PIDESELO al usuario y espera su respuesta; nunca registres con
  datos incompletos ni sin que el usuario confirme explicitamente (confirmar=True solo
  despues de que el usuario diga que si).
- Si ninguna tool de conocimiento devuelve informacion relevante, responde exactamente:
  "No encontre informacion suficiente en la base documental proporcionada." No inventes
  datos que no esten en el contexto recuperado.
- Al final de cada respuesta, agrega una linea "Agentes utilizados: ..." indicando que
  tool(s) invocaste, para dar trazabilidad."""


# Memoria: permite conversaciones multi-turno 

memoria = InMemorySaver()
orquestador = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria,
)

print("Tools registradas en el orquestador:")
for t in tools_orquestador:
    print("  -", t.name)


def _imprimir_pasos(resultado):
    """Muestra que tools se invocaron y su resultado (trazabilidad)."""
    for m in resultado["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print(f"[TOOL] {tc['name']}({tc['args']})")
        if m.__class__.__name__ == "ToolMessage":
            print(f"[RESPONSE] {str(m.content)[:300]}\n")


def extraer_texto(content):
    """Gemini a veces devuelve el content como una LISTA de bloques
    (texto + firmas de 'thinking'). Esta funcion devuelve solo el texto plano."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        partes = []
        for b in content:
            if isinstance(b, dict):
                partes.append(b.get("text", ""))
            elif isinstance(b, str):
                partes.append(b)
        return "".join(partes).strip()
    return str(content)


def consultar(pregunta: str, thread_id: str = None):
    """Invoca al orquestador (una consulta suelta) e imprime tools + respuesta final."""
    thread_id = thread_id or f"demo-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}
    print(f">>> Usuario: {pregunta}\n")
    resultado = orquestador.invoke({"messages": [{"role": "user", "content": pregunta}]}, config)
    _imprimir_pasos(resultado)
    print("=== Respuesta final ===")
    print(extraer_texto(resultado["messages"][-1].content))
    return resultado
if __name__ == "__main__":
    # Prueba 1: agente de conocimiento simple
    consultar("¿Cuántos días de vacaciones me corresponden al año?")
 
    # Prueba 2: consulta mixta (vacaciones + beneficios) -> obliga a invocar 2 tools
    consultar(
        "Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro médico. "
        "¿Cuántos días me corresponden, cómo los solicito y qué necesito para inscribir "
        "a un dependiente en el beneficio?"
    )
 
    # Prueba 3: multimodal
    consultar("Adjunto el formulario en formulario_dependiente.png: ¿está completo y qué datos faltan?")
 
    # Prueba 4: registrar con datos incompletos -> el sistema de control debe pedir lo que falta
    consultar("Registra una solicitud de vacaciones para Juan Perez.")
 
    # Prueba 5: fuera de alcance -> debe admitir que no tiene informacion
    consultar("¿Cuál es el precio de las acciones de Patito S.A. en la bolsa?")
    

El archivo ya existe. Procediendo a la lectura...
Caracteres totales: 1302
                                                            
PATITO S.A.
MANUAL DE BENEFICIOS Y COMPENSACIONES
(Documento ficticio para fines de evaluación del semillero)
Base de conocimiento del Agente de Beneficios y Compensaciones

1. SEGURO MÉDICO CORPORATIVO
1.1 Cobertura: consultas médicas, hospitalización, emergencias, exámenes de laboratorio y
    medicamentos según el plan. Incluye atención ambulatoria y cobertura dental básica.
1.2 Dependientes: e
Total de chunks creados: 4
Chunk 1: 1. SEGURO MÉDICO CORPORATIVO 1.1 Cobertura: consultas médicas, hospita...
Chunk 2: 2. BONOS - Bono por desempeño anual según evaluación. - Bono por cumpl...
Chunk 3: 3. OTROS BENEFICIOS - Día libre de cumpleaños. - Capacitación y apoyo ...
Chunk 4: 4. COMPENSACIÓN La estructura salarial considera el rol, la banda sala...
El archivo 02_Reglamento_Interno.txt ya existe. 
Caracteres totales: 1441
                              